# 09 - Statistical Tests and Final Report

## Goal
- compare three top ensembles from notebook 08,
- run statistical tests for pairwise differences,
- apply multiple-comparison correction,
- produce final statistical report for project conclusion.


In [ ]:
import itertools
import json
import os

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import chi2
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


In [ ]:
try:
    from stylin import InfoDisplayStyler
    styler = InfoDisplayStyler()
except Exception:
    class _FallbackStyler:
        def style_me(self, obj, title=None):
            if title:
                print(f"\n=== {title} ===")
            display(obj)

        def show_line(self, *args, sep=' ', title=None):
            text = sep.join(str(a) for a in args).strip()
            if title:
                print(f"{title}: {text}")
            else:
                print(text)

        def show_meta(self, data):
            print('shape:', getattr(data, 'shape', None))
            display(data.head(3) if hasattr(data, 'head') else data)

    styler = _FallbackStyler()


In [ ]:
DATA_PATH = '../../data/raw/ethusdt_1h.csv'
CLEAN_PATH = '../../data/clean/'
LABELED_PATH = '../../data/labeled/'
PROCESSED_PATH = '../../data/processed/'
REPORTS_PATH = '../../reports/'

TOP3_METRICS_FILE = REPORTS_PATH + 'top3_ensemble_metrics.csv'
TOP3_SUMMARY_FILE = REPORTS_PATH + 'top3_ensemble_summary.json'
PREDICTIONS_DIR = REPORTS_PATH + 'predictions/'

assert os.path.exists(TOP3_METRICS_FILE), 'Missing top3_ensemble_metrics.csv. Run notebook 08 first.'
assert os.path.exists(TOP3_SUMMARY_FILE), 'Missing top3_ensemble_summary.json. Run notebook 08 first.'
assert os.path.exists(PREDICTIONS_DIR), 'Missing reports/predictions/. Run notebook 08 first.'

metrics_df = pd.read_csv(TOP3_METRICS_FILE).sort_values('rank').reset_index(drop=True)
with open(TOP3_SUMMARY_FILE, 'r', encoding='utf-8') as f:
    top3_summary = json.load(f)

run_id = top3_summary['run_id']

styler.style_me(metrics_df, title='Top-3 ensemble metrics from notebook 08')
styler.show_line(run_id, title='Run ID used for statistical tests')


## 1. Load ensemble predictions (3 sets) and align timestamps


In [ ]:
def load_ensemble_predictions(rank: int, run_id: str) -> pd.DataFrame:
    path = REPORTS_PATH + f'predictions/run_{run_id}_rank{rank}_ensemble_test_predictions.csv'
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing prediction file for rank {rank}: {path}')
    df = pd.read_csv(path)
    required = ['timestamp', 'y_true', 'y_pred', 'prob_sell', 'prob_hold', 'prob_buy']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'File {path} missing columns: {missing}')
    return df

preds = {}
for rank in [1, 2, 3]:
    preds[rank] = load_ensemble_predictions(rank, run_id)

# Align by timestamp to guarantee same sample order.
base_ts = preds[1]['timestamp'].astype(str)
for rank in [2, 3]:
    merged = pd.merge(
        preds[1][['timestamp']],
        preds[rank],
        on='timestamp',
        how='left',
    )
    assert merged['y_pred'].notna().all(), f'Rank {rank} missing timestamps after alignment'
    preds[rank] = merged

alignment_report = pd.DataFrame({
    'rank': [1, 2, 3],
    'rows': [len(preds[1]), len(preds[2]), len(preds[3])],
    'unique_timestamps': [preds[1]['timestamp'].nunique(), preds[2]['timestamp'].nunique(), preds[3]['timestamp'].nunique()],
})
styler.style_me(alignment_report, title='Prediction alignment report')


## 2. Helper functions: bootstrap, McNemar, Holm correction


In [ ]:
def bootstrap_metric_difference(
    y_true: np.ndarray,
    y_pred_a: np.ndarray,
    y_pred_b: np.ndarray,
    metric_fn,
    n_bootstrap: int = 2000,
    random_state: int = 42,
):
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    diffs = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        score_a = metric_fn(y_true[idx], y_pred_a[idx])
        score_b = metric_fn(y_true[idx], y_pred_b[idx])
        diffs.append(score_a - score_b)

    diffs = np.array(diffs)
    return {
        'mean_diff': float(np.mean(diffs)),
        'ci_lower_95': float(np.quantile(diffs, 0.025)),
        'ci_upper_95': float(np.quantile(diffs, 0.975)),
    }


def mcnemar_test(y_true: np.ndarray, y_pred_a: np.ndarray, y_pred_b: np.ndarray):
    correct_a = (y_pred_a == y_true)
    correct_b = (y_pred_b == y_true)

    n01 = int(np.sum((correct_a == 1) & (correct_b == 0)))  # A correct, B wrong
    n10 = int(np.sum((correct_a == 0) & (correct_b == 1)))  # A wrong, B correct

    # If n01+n10 == 0, predictions are identical wrt correctness.
    if n01 + n10 == 0:
        return {'n01': n01, 'n10': n10, 'chi2': 0.0, 'p_value': 1.0}

    # Continuity-corrected McNemar statistic.
    chi2_stat = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
    p_value = float(1 - chi2.cdf(chi2_stat, df=1))
    return {'n01': n01, 'n10': n10, 'chi2': float(chi2_stat), 'p_value': p_value}


def holm_correction(pvals: list[float]) -> list[float]:
    m = len(pvals)
    order = np.argsort(pvals)
    adjusted = np.empty(m, dtype=float)

    running_max = 0.0
    for i, idx in enumerate(order):
        adj = (m - i) * pvals[idx]
        running_max = max(running_max, adj)
        adjusted[idx] = min(1.0, running_max)

    return adjusted.tolist()


## 3. Pairwise statistical comparison between 3 ensembles


In [ ]:
y_true = preds[1]['y_true'].to_numpy(dtype=np.int64)

pairs = [(1, 2), (1, 3), (2, 3)]
rows = []
mcnemar_pvals = []
mcnemar_rows_cache = []

for a, b in pairs:
    y_pred_a = preds[a]['y_pred'].to_numpy(dtype=np.int64)
    y_pred_b = preds[b]['y_pred'].to_numpy(dtype=np.int64)

    bs_f1 = bootstrap_metric_difference(
        y_true=y_true,
        y_pred_a=y_pred_a,
        y_pred_b=y_pred_b,
        metric_fn=lambda yt, yp: float(f1_score(yt, yp, average='macro', zero_division=0)),
    )
    bs_bal = bootstrap_metric_difference(
        y_true=y_true,
        y_pred_a=y_pred_a,
        y_pred_b=y_pred_b,
        metric_fn=lambda yt, yp: float(balanced_accuracy_score(yt, yp)),
    )
    bs_acc = bootstrap_metric_difference(
        y_true=y_true,
        y_pred_a=y_pred_a,
        y_pred_b=y_pred_b,
        metric_fn=lambda yt, yp: float(accuracy_score(yt, yp)),
    )

    mc = mcnemar_test(y_true=y_true, y_pred_a=y_pred_a, y_pred_b=y_pred_b)
    mcnemar_pvals.append(mc['p_value'])
    mcnemar_rows_cache.append((a, b, mc))

    rows.append({
        'pair': f'rank{a}_vs_rank{b}',
        'f1_mean_diff': bs_f1['mean_diff'],
        'f1_ci_low': bs_f1['ci_lower_95'],
        'f1_ci_high': bs_f1['ci_upper_95'],
        'balacc_mean_diff': bs_bal['mean_diff'],
        'balacc_ci_low': bs_bal['ci_lower_95'],
        'balacc_ci_high': bs_bal['ci_upper_95'],
        'acc_mean_diff': bs_acc['mean_diff'],
        'acc_ci_low': bs_acc['ci_lower_95'],
        'acc_ci_high': bs_acc['ci_upper_95'],
        'mcnemar_n01': mc['n01'],
        'mcnemar_n10': mc['n10'],
        'mcnemar_chi2': mc['chi2'],
        'mcnemar_p': mc['p_value'],
    })

stat_df = pd.DataFrame(rows)
stat_df


## 4. Multiple-comparison correction (Holm)


In [ ]:
adj = holm_correction(mcnemar_pvals)

for i, p_adj in enumerate(adj):
    stat_df.loc[i, 'mcnemar_p_holm'] = p_adj
    stat_df.loc[i, 'mcnemar_significant_0_05'] = bool(p_adj < 0.05)

# For bootstrap CI: significance if CI does not include 0.
stat_df['f1_diff_significant_by_ci'] = ~((stat_df['f1_ci_low'] <= 0) & (stat_df['f1_ci_high'] >= 0))
stat_df['balacc_diff_significant_by_ci'] = ~((stat_df['balacc_ci_low'] <= 0) & (stat_df['balacc_ci_high'] >= 0))
stat_df['acc_diff_significant_by_ci'] = ~((stat_df['acc_ci_low'] <= 0) & (stat_df['acc_ci_high'] >= 0))

styler.style_me(stat_df, title='Pairwise statistical tests (with Holm correction)')


## 5. Final ranking summary


In [ ]:
rank_metrics = metrics_df[['rank', 'trial_number', 'test_f1_macro', 'test_balanced_accuracy', 'test_accuracy']].copy()
rank_metrics = rank_metrics.sort_values('test_f1_macro', ascending=False).reset_index(drop=True)
rank_metrics['order_by_test_f1'] = np.arange(1, len(rank_metrics) + 1)

styler.style_me(rank_metrics, title='Ensemble ranking by test f1_macro')


## 6. Save statistical report artifacts


In [ ]:
STAT_CSV = REPORTS_PATH + 'stat_tests_summary.csv'
STAT_JSON = REPORTS_PATH + 'stat_tests_summary.json'
RANKING_CSV = REPORTS_PATH + 'final_ensemble_ranking.csv'

stat_df.to_csv(STAT_CSV, index=False)
rank_metrics.to_csv(RANKING_CSV, index=False)

payload = {
    'run_id': run_id,
    'pairs_tested': stat_df.to_dict(orient='records'),
    'ranking_by_test_f1': rank_metrics.to_dict(orient='records'),
    'notes': {
        'bootstrap_metric_diff': 'Difference defined as metric(rankA) - metric(rankB).',
        'mcnemar': 'Continuity-corrected McNemar test on correctness disagreement table.',
        'multiple_comparison_correction': 'Holm correction over 3 pairwise McNemar p-values.',
    },
}
with open(STAT_JSON, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

styler.show_line(STAT_CSV, title='Saved statistical summary CSV')
styler.show_line(STAT_JSON, title='Saved statistical summary JSON')
styler.show_line(RANKING_CSV, title='Saved final ranking CSV')


## 7. Final conclusions (fill after run)
- Which ensemble is best on test metrics?
- Are differences between ensembles statistically significant after Holm correction?
- Do bootstrap confidence intervals support the same conclusion?
- This notebook closes point 10 of the assignment.
